# Evaluation of Whisper Models

## 1 Overview

## 2 Importing Libraries

In [1]:
from pathlib import Path
import gc
import re
import time
import numpy as np
import pandas as pd
import torch
from transformers import pipeline

c:\Users\Subathra\OneDrive\Desktop\cm3020_Final_Year_project\CM3020_Final_Year_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3 Evaluation Settings and Candidate Models

In [2]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = 0 if torch.cuda.is_available() else -1

In [3]:
manifest_folder = Path(
    "../../data/processed/ravdess"
)

validation_manifest = (
    manifest_folder / "validation.csv"
)

testing_manifest = (
    manifest_folder / "testing.csv"
)

raw_audio_folder = Path(
    "../../data/raw/ravdess"
)

output_folder = Path("outputs/whisper")

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

In [4]:
models = {
    "Whisper-Tiny": {
        "model_id": "openai/whisper-tiny.en",
        "batch_size": 16
    },

    "Whisper-Base": {
        "model_id": "openai/whisper-base.en",
        "batch_size": 8
    },

    "Whisper-Small": {
        "model_id": "openai/whisper-small.en",
        "batch_size": 4
    }
}

## 4 Load RAVDESS Dataset

In [5]:
def load_manifest(manifest_path):
    data = pd.read_csv(manifest_path)

    required_columns = {
        "filename",
        "actor",
        "statement"
    }

    missing_columns = (
        required_columns - set(data.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Missing columns: {sorted(missing_columns)}"
        )

    data = data.copy()

    data["actor_number"] = (
        data["actor"]
        .astype(str)
        .str.extract(r"(\d+)")[0]
        .astype(int)
    )

    data["audio_path"] = data.apply(
        lambda row: (
            raw_audio_folder
            / f"Actor_{row['actor_number']:02d}"
            / str(row["filename"])
        ),
        axis=1
    )

    missing_files = (
        ~data["audio_path"].apply(
            lambda path: path.is_file()
        )
    )

    if missing_files.any():
        raise FileNotFoundError(
            f"{missing_files.sum()} audio files "
            f"could not be located."
        )

    return data.reset_index(drop=True)

In [6]:
validation_data = load_manifest(
    validation_manifest
)

testing_data = load_manifest(
    testing_manifest
)

print("Validation recordings:", len(validation_data))
print("Testing recordings:", len(testing_data))

Validation recordings: 960
Testing recordings: 480


## 5 Reference Transcripts

In [7]:
statement_mapping = {
    1: "kids are talking by the door",
    2: "dogs are sitting by the door"
}

In [8]:
def normalise_text(text):
    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        "",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [9]:
def add_reference_transcripts(data):
    prepared_data = data.copy()

    prepared_data["statement_number"] = (
        pd.to_numeric(
            prepared_data["statement"]
        ).astype(int)
    )

    prepared_data["reference"] = (
        prepared_data["statement_number"]
        .map(statement_mapping)
        .apply(normalise_text)
    )

    if prepared_data["reference"].isna().any():
        raise ValueError(
            "Unknown RAVDESS statement code."
        )

    return prepared_data

In [10]:
validation_data = add_reference_transcripts(
    validation_data
)

testing_data = add_reference_transcripts(
    testing_data
)

display(
    validation_data[
        ["filename", "reference"]
    ].head()
)

,filename,reference
0,03-01-01-01-01-01-01.wav,kids are talking by the door
1,03-01-01-01-01-02-01.wav,kids are talking by the door
2,03-01-01-01-02-01-01.wav,dogs are sitting by the door
3,03-01-01-01-02-02-01.wav,dogs are sitting by the door
4,03-01-02-01-01-01-01.wav,kids are talking by the door


## 6 Transcription and WER functions

In [11]:
def word_error_counts(reference, hypothesis):
    reference_words = normalise_text(
        reference
    ).split()

    hypothesis_words = normalise_text(
        hypothesis
    ).split()

    rows = len(reference_words) + 1
    columns = len(hypothesis_words) + 1

    distance = [
        [0] * columns
        for _ in range(rows)
    ]

    for row in range(rows):
        distance[row][0] = row

    for column in range(columns):
        distance[0][column] = column

    for row in range(1, rows):
        for column in range(1, columns):
            substitution_cost = (
                0
                if reference_words[row - 1]
                == hypothesis_words[column - 1]
                else 1
            )

            distance[row][column] = min(
                distance[row - 1][column] + 1,
                distance[row][column - 1] + 1,
                distance[row - 1][column - 1]
                + substitution_cost
            )

    return (
        distance[-1][-1],
        len(reference_words)
    )

In [12]:
def calculate_asr_metrics(
    references,
    transcriptions
):
    total_errors = 0
    total_words = 0
    exact_matches = 0
    sentence_wers = []

    for reference, transcription in zip(
        references,
        transcriptions
    ):
        errors, words = word_error_counts(
            reference,
            transcription
        )

        total_errors += errors
        total_words += words

        sentence_wers.append(
            errors / words if words else 0
        )

        if normalise_text(reference) == normalise_text(
            transcription
        ):
            exact_matches += 1

    return {
        "WER": (
            total_errors / total_words
            if total_words
            else 0
        ),
        "Average Sentence WER": np.mean(
            sentence_wers
        ),
        "Exact Match Accuracy": (
            exact_matches / len(references)
        )
    }

In [13]:
def run_whisper_model(model_details, data):
    model_id = model_details["model_id"]
    batch_size = model_details["batch_size"]

    loading_start = time.perf_counter()

    transcriber = pipeline(
        task="automatic-speech-recognition",
        model=model_id,
        device=device
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    loading_time = (
        time.perf_counter() - loading_start
    )

    audio_paths = (
        data["audio_path"]
        .astype(str)
        .tolist()
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    inference_start = time.perf_counter()

    outputs = transcriber(
        audio_paths,
        batch_size=batch_size
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    inference_time = (
        time.perf_counter() - inference_start
    )

    transcriptions = [
        normalise_text(output["text"])
        for output in outputs
    ]

    del transcriber

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "transcriptions": transcriptions,
        "loading_time": loading_time,
        "inference_time": inference_time
    }

## 7 Model Evaluation on Validation Data

In [14]:
validation_results = []
validation_transcriptions = {}

validation_references = (
    validation_data["reference"].tolist()
)

for model_name, model_details in models.items():
    print(f"Evaluating {model_name}...")

    evaluation = run_whisper_model(
        model_details,
        validation_data
    )

    transcriptions = evaluation[
        "transcriptions"
    ]

    metrics = calculate_asr_metrics(
        validation_references,
        transcriptions
    )

    validation_transcriptions[model_name] = (
        transcriptions
    )

    validation_results.append({
        "Model": model_name,
        **metrics,
        "Loading Time": evaluation["loading_time"],
        "Inference Time": evaluation["inference_time"],
        "ms per Recording": (
            evaluation["inference_time"]
            / len(validation_data)
            * 1000
        )
    })

    print(
        "WER:",
        round(metrics["WER"], 4)
    )

Evaluating Whisper-Tiny...


c:\Users\Subathra\OneDrive\Desktop\cm3020_Final_Year_project\CM3020_Final_Year_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Subathra\.cache\huggingface\hub\models--openai--whisper-tiny.en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 167/167 [00:00<0

WER: 0.0158
Evaluating Whisper-Base...


c:\Users\Subathra\OneDrive\Desktop\cm3020_Final_Year_project\CM3020_Final_Year_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Subathra\.cache\huggingface\hub\models--openai--whisper-base.en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 245/245 [00:00<0

WER: 0.0075
Evaluating Whisper-Small...


c:\Users\Subathra\OneDrive\Desktop\cm3020_Final_Year_project\CM3020_Final_Year_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Subathra\.cache\huggingface\hub\models--openai--whisper-small.en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 479/479 [00:00<

WER: 0.0023


In [15]:
validation_results_df = pd.DataFrame(
    validation_results
)

display(validation_results_df)

,Model,WER,Average Sentence WER,Exact Match Accuracy,Loading Time,Inference Time,ms per Recording
0,Whisper-Tiny,0.015799,0.015799,0.939583,14.017245,97.048086,101.091756
1,Whisper-Base,0.007465,0.007465,0.965625,12.348293,135.197844,140.831088
2,Whisper-Small,0.002257,0.002257,0.987500,22.260110,309.368200,322.258542


## 8 Comparing and Selecting Best Model

In [16]:
validation_results_df = (
    validation_results_df
    .sort_values(
        by=[
            "WER",
            "Exact Match Accuracy",
            "Inference Time"
        ],
        ascending=[True, False, True]
    )
    .reset_index(drop=True)
)

display(validation_results_df)

,Model,WER,Average Sentence WER,Exact Match Accuracy,Loading Time,Inference Time,ms per Recording
0,Whisper-Small,0.002257,0.002257,0.987500,22.260110,309.368200,322.258542
1,Whisper-Base,0.007465,0.007465,0.965625,12.348293,135.197844,140.831088
2,Whisper-Tiny,0.015799,0.015799,0.939583,14.017245,97.048086,101.091756


In [17]:
selected_model_name = (
    validation_results_df.loc[0, "Model"]
)

selected_model_details = models[
    selected_model_name
]

print("Selected model:", selected_model_name)

print(
    "Validation WER:",
    round(
        validation_results_df.loc[0, "WER"],
        4
    )
)

Selected model: Whisper-Small
Validation WER: 0.0023


## 9 Final Test Evaluation

In [18]:
test_evaluation = run_whisper_model(
    selected_model_details,
    testing_data
)

test_transcriptions = test_evaluation[
    "transcriptions"
]

test_references = (
    testing_data["reference"].tolist()
)

test_metrics = calculate_asr_metrics(
    test_references,
    test_transcriptions
)

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 8112.54it/s]


In [19]:
test_results_df = pd.DataFrame([{
    "Model": selected_model_name,
    **test_metrics,
    "Loading Time": test_evaluation["loading_time"],
    "Inference Time": test_evaluation["inference_time"],
    "ms per Recording": (
        test_evaluation["inference_time"]
        / len(testing_data)
        * 1000
    )
}])

display(test_results_df)

,Model,WER,Average Sentence WER,Exact Match Accuracy,Loading Time,Inference Time,ms per Recording
0,Whisper-Small,0.004167,0.004167,0.977083,4.306587,160.276003,333.90834


## 10 Transcription Error Analysis

In [20]:
test_predictions_df = testing_data[
    ["filename", "actor", "reference"]
].copy()

test_predictions_df["transcription"] = (
    test_transcriptions
)

test_predictions_df["Sentence WER"] = [
    word_error_counts(reference, transcription)[0]
    / word_error_counts(reference, transcription)[1]
    for reference, transcription in zip(
        test_references,
        test_transcriptions
    )
]

display(
    test_predictions_df
    .sort_values(
        "Sentence WER",
        ascending=False
    )
    .head(20)
)

,filename,actor,reference,transcription,Sentence WER
166,03-01-07-01-02-01-19.wav,19,dogs are sitting by the door,dogsit sitting by the door,0.333333
18,03-01-03-02-02-01-17.wav,17,dogs are sitting by the door,the dogs are sitting by the door,0.166667
458,03-01-06-01-02-01-24.wav,24,dogs are sitting by the door,talks are sitting by the door,0.166667
207,03-01-04-02-02-02-20.wav,20,dogs are sitting by the door,the dogs are sitting by the door,0.166667
214,03-01-05-02-02-01-20.wav,20,dogs are sitting by the door,the dogs are sitting by the door,0.166667
78,03-01-03-02-02-01-18.wav,18,dogs are sitting by the door,dolls are sitting by the door,0.166667
217,03-01-06-01-01-02-20.wav,20,kids are talking by the door,kids are dogging by the door,0.166667
219,03-01-06-01-02-02-20.wav,20,dogs are sitting by the door,talks are sitting by the door,0.166667
206,03-01-04-02-02-01-20.wav,20,dogs are sitting by the door,talks are sitting by the door,0.166667
278,03-01-06-01-02-01-21.wav,21,dogs are sitting by the door,dogs are sitting behind the door,0.166667


## 11 Saving Results

In [21]:
validation_results_df.to_csv(
    output_folder / "whisper_model_comparison.csv",
    index=False
)

test_results_df.to_csv(
    output_folder
    / "selected_whisper_model_test_results.csv",
    index=False
)

test_predictions_df.to_csv(
    output_folder
    / "selected_whisper_model_transcriptions.csv",
    index=False
)

print("Results saved in:", output_folder)

Results saved in: outputs\whisper


## 12 Conclusion